<a href="https://colab.research.google.com/github/jacobwechuli/mypython/blob/main/PDF_Retrieval_and_Optimization_with_LlamaIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q llama-index llama-index-llms-gemini pymupdf
!pip install -q llama-index-embeddings-huggingface
!pip install -q nest_asyncio
!pip install -q llama-index-retrievers-bm25
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import nest_asyncio

nest_asyncio.apply()

# Set up Google API key for Gemini
GOOGLE_API_KEY = "//"  # Replace with your actual API key
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

if GOOGLE_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ WARNING: Please replace YOUR_API_KEY_HERE with your actual API key!")
    print("Get one free at: https://aistudio.google.com/app/apikey")

# Create a directory for our PDFs if it doesn't exist
!mkdir -p sample_docs

In [3]:
import fitz  # PyMuPDF

# Load PDF document
doc = fitz.open("sample_docs/contract.pdf")

# Extract text from all pages
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the PDF.")

FileNotFoundError: no such file: 'sample_docs/contract.pdf'

In [ ]:
from llama_index.llms.gemini import Gemini
from llama_index.core.llms import ChatMessage

# Set up Gemini API key
import os
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY"

# Initialize Gemini LLM
llm = Gemini(model="models/gemini-2.0-flash")

# Define query rewriting function
def rewrite_query(user_query):
    messages = [
        ChatMessage(role="system", content="Rewrite this query for improved retrieval relevance."),
        ChatMessage(role="user", content=user_query),
    ]
    response = llm.chat(messages)
    return response.message.content

# Test query rewriting
query = "What are the penalties for late payments?"
expanded_query = rewrite_query(query)

print(f"Original Query: {query}")
print(f"Expanded Query: {expanded_query}")

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import BaseRetriever

def create_hybrid_retriever(index, query, top_k=2):
    """Create a hybrid retrieval approach combining vector and keyword search."""
    # Method 1: Vector retrieval (semantic search)
    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    vector_nodes = vector_retriever.retrieve(query)

    # Method 2: BM25 retrieval (keyword-based search)
    nodes = [node for node in index.docstore.docs.values()]
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=nodes,
        similarity_top_k=top_k
    )
    keyword_nodes = bm25_retriever.retrieve(query)

    # Combine results
    all_nodes = list(vector_nodes) + list(keyword_nodes)

    # Remove duplicates by node_id
    unique_nodes = {}
    for node in all_nodes:
        if node.node_id not in unique_nodes:
            unique_nodes[node.node_id] = node

    # Sort by score (higher is better)
    sorted_nodes = sorted(
        unique_nodes.values(),
        key=lambda x: x.score if hasattr(x, 'score') else 0.0,
        reverse=True
    )

    return sorted_nodes[:top_k]

# Test hybrid retrieval
hybrid_nodes = create_hybrid_retriever(index, "What is the refund policy?")
for i, node in enumerate(hybrid_nodes):
    print(f"Result {i+1} (Score: {node.score:.4f}):")
    print(node.get_text()[:200])
    print("-" * 40)from llama_index.core.postprocessor import SentenceTransformerRerank

# Create a reranker using a cross-encoder model
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=2
)

# First, retrieve results
retriever = index.as_retriever(similarity_top_k=3)
query = "What is the refund policy?"
retrieved_nodes = retriever.retrieve(query)

print("Before Reranking:")
for i, node in enumerate(retrieved_nodes):
    print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")

# Apply reranking
reranked_nodes = reranker.postprocess_nodes(
    retrieved_nodes,
    query_str=query
)

print("\nAfter Reranking:")
for i, node in enumerate(reranked_nodes):
    print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")